01:
YOLO檢測->追蹤車輛 把相關資訊存成csv

In [ ]:
!pip install ultralytics #YOLO的Python package

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt") #建YOLO(model object) 載入預訓練的yolo11n.pt(權重)

In [ ]:
#確認影片是否成功載入
import os
print(os.listdir("/content"))

In [ ]:
#以YOLO預測影片物件-影片存到本機-確認是否成功框出事故車(手動)
results = model.predict(
    source="/content/1.mp4",
    save=True,
    conf=0.25
)

In [ ]:
#理解YOLO的output-拿影片的其中一個frame-看有什麼參數
r = results[0]
print(r.boxes)

In [ ]:
#detecting OK -> tracking
tracking_results = model.track(
    source="/content/1.mp4",
    tracker="bytetrack.yaml",
    persist=True,
    save=True,
    project="/content/tracking",
    conf=0.25
)

In [ ]:
#同上 看tracking的第一個frame內有什麼參數
r = tracking_results[0]
print(r.boxes)
#print(r.boxes.xyxy)
#print(r.boxes.conf)
#print(r.boxes.cls)
#print(r.boxes.id)

In [ ]:
#看tracking後的video 確認影片沒問題(手動)
"""
把tracking後 每一個frame內的每個bounding box
除了原本的相關資訊外 再額外加上加兩欄 center_x center_y (之後畫trajectory)
存成dataframe
"""

import pandas as pd
rows = []

for frame_idx, r in enumerate(tracking_results):

    # 這個frame沒有detection
    if r.boxes is None or r.boxes.id is None:
        continue

    # 這個frame有detection:把該frame中偵測到的每個bounding box的相關資訊(座標/conf/class/id)拆成各類的object
    boxes = r.boxes.xyxy.cpu().numpy()
    confidences = r.boxes.conf.cpu().numpy()
    classes = r.boxes.cls.cpu().numpy()
    track_ids = r.boxes.id.cpu().numpy()

    # 遍歷該frame中每個bounding box並進行資料處理 接著存到row內
    for box, conf, cls, track_id in zip(boxes, confidences, classes, track_ids):
        x1, y1, x2, y2 = box

        center_x = (x1 + x2) / 2
        center_y = (y1 + y2) / 2

        rows.append({
            "frame": frame_idx,
            "track_id": int(track_id),
            "class_id": int(cls),
            "confidence": float(conf),
            "x1": float(x1),
            "y1": float(y1),
            "x2": float(x2),
            "y2": float(y2),
            "center_x": float(center_x),
            "center_y": float(center_y)
        })

#row內有每個frame中 每個bounding box的相關資訊
#把row存成dataframe
df = pd.DataFrame(rows)

#確認資訊無誤
df.head(20)

In [ ]:
#檢查df
print(df.shape)
print(df["track_id"].unique())
print(df.head())

In [ ]:
#確認df沒問題 轉csv
df.to_csv("/content/tracking/tracking_result_1.csv", index=False)